# Batch integration of a cohort's scRNA with Harmony and scVI

We grow a small cohort of patients, measure their dissociated cells by single-cell RNA sequencing (scRNA-seq) in sequencing batches, and integrate the batches with Harmony ([Korsunsky et al. 2019](https://github.com/slowkow/harmonypy)) and scVI ([Lopez et al. 2018](https://scvi-tools.org)), both through scanpy.
We pool the same cells into batches in two designs, one batch per patient and patients spread across batches, and score each integration against the answer key by how well it mixes the batches and by how much of the patient-specific clonal structure of the cancer cells it keeps.
[The cohort tutorial](04_cohort.ipynb) shows how a cohort is grown and what its answer key holds.

Harmony and scVI run in their own conda environment, `iscc-integration` (scanpy, harmonypy and scvi-tools on CPU), from a short script that the notebook writes.

In [1]:
import os
import tempfile
import warnings

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

from iscc.cohort import Cohort, concat_cohort_batches, pool_cell_data, run_cohort_batches
from iscc.data import concat_batches, run_scrna_batches, scRNA
from iscc.integrations import clones_from_clades
from iscc.sample import Resection
from iscc.tumor import GenotypeTumor

CONFIG = os.path.abspath("tutorial_config.yaml")
INTEGRATION_PYTHON = os.path.expanduser("~/miniconda3/envs/iscc-integration/bin/python")
os.chdir(tempfile.mkdtemp())        # the data and the tool's outputs go to a scratch directory
warnings.filterwarnings("ignore", message="max_cells=")   # we dissociate a fixed number of cells on purpose

## A cohort of six patients

[`Cohort`](../reference/Cohort.md) grows each patient from its own evolution seed over one gene layout, so the patients share their drivers, baseline expression and programs and differ in how their tumors evolved.
It calls a growth function that we write, which builds a [`GenotypeTumor`](../reference/GenotypeTumor.md) from the tutorial configuration with the epithelium as the cell of origin, grows it to 20,000 cancer cells, and dissociates the resected tumor into 1,000 cells, of which each cell type is recovered with its default probability.
The dissociated cells become the tumor's own cell data (`install=True`), which the cohort's batching functions read.

In [2]:
def grow(seed, layout_seed):
    tumor = GenotypeTumor(config=CONFIG, seed=seed, layout_seed=layout_seed,
                          expression_params={"cell_of_origin": "epithelial"})
    tumor.grow(n_steps=3000, target_cancer=20_000, materialize=False)
    Resection(tumor).dissociate(max_cells=1000, recovery_probs="default", seed=seed, install=True)
    return tumor


cohort = Cohort(grow, patient_seeds=range(1, 7)).run()
pooled, meta = pool_cell_data(cohort.patients)
pd.crosstab(meta["patient"], meta["cell_type"], margins=True)

cell_type,cancer,epithelial,stromal,All
patient,,,,
0,180,41,427,648
1,179,43,447,669
2,167,57,472,696
3,171,53,435,659
4,203,47,472,722
5,184,43,445,672
All,1084,284,2698,4066


In the answer key, every cell has its patient and cell type, and the true genealogy of each tumor defines its clones.
We call a clone each clade of the genealogy that holds at least 20 of the patient's sampled cells ([`clones_from_clades`](../reference/clones_from_clades.md)), and name it by its patient.

In [3]:
clones = pd.concat([clones_from_clades(p.tumor, min_cells=20).astype(str).rename(lambda c, p=p: f"P{p.patient_id}::{c}")
                    for p in cohort.patients])
meta["lineage"] = "P" + meta["patient"].astype(str) + ":" + clones.reindex(meta.index).to_numpy()
meta.loc[meta["cell_type"] == "cancer"].groupby("patient")["lineage"].nunique().rename("clones").to_frame().T

patient,0,1,2,3,4,5
clones,2,3,4,4,4,3


## Two pooling designs

We measure the cohort with the 10x Genomics preset of [`scRNA`](../reference/scRNA.md) in six batches, each its own realization of the batch effect (a per-gene factor, depth, ambient RNA and doublets).
In the first design each patient is its own batch, as when patients are sequenced one at a time ([`run_cohort_batches`](../reference/Cohort.md)), so the batch and the patient are confounded.
In the second design the pooled cells are split at random into six batches, each holding every patient, as when patients are multiplexed ([`run_scrna_batches`](../reference/scRNA.md)).
The same cells measured in one batch without any batch effect give the reference, the structure an integration should keep.

In [4]:
def to_h5ad(assays, name):
    for a in assays:                                   # the answer key, for scoring only
        a.obs[["patient", "cell_type", "lineage"]] = meta.loc[a.obs.index, ["patient", "cell_type", "lineage"]].to_numpy()
    adata = concat_batches(assays)
    adata.obs = adata.obs.astype({"patient": str})
    adata.write_h5ad(f"{name}.h5ad")
    return adata


by_patient, _, _ = run_cohort_batches(cohort, mapping="one_to_one", n_cells_per_patient=None, protocol="10x")
across = run_scrna_batches(pooled, n_batches=6, design="split", n_cells=len(meta) // 6, protocol="10x")
reference = scRNA(protocol="10x", n_cells=len(meta), sigma_batch=0.0, depth_batch_sigma=0.0, seed=7).run(pooled)
data = {"by patient": to_h5ad(by_patient, "by_patient"), "across patients": to_h5ad(across, "across_patients"),
        "reference": to_h5ad([reference], "reference")}
for name, adata in data.items():
    print(f"{name:>15}: {adata.n_obs:,} cells in {adata.obs['batch'].nunique()} batches")

     by patient: 4,066 cells in 6 batches
across patients: 4,062 cells in 6 batches
      reference: 4,066 cells in 1 batches


## Harmony and scVI

The script below runs the standard workflows of scanpy and scvi-tools on each data set.
It selects 2,000 highly variable genes on the counts across batches, normalizes and log-transforms them, and computes 50 principal components, which are the embedding without integration.
Harmony corrects the principal components for the batch (`sc.external.pp.harmony_integrate`), and scVI learns a latent space from the counts with the batch as a covariate, both at their defaults.

In [5]:
%%writefile integrate.py
import sys

import numpy as np
import scanpy as sc
import scvi


def main(name):
    adata = sc.read_h5ad(f"{name}.h5ad")
    adata.layers["counts"] = adata.X.copy()
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", layer="counts",
                                batch_key="batch", subset=True)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.pca(adata, n_comps=50)
    embeddings = {"none": adata.obsm["X_pca"]}

    if adata.obs["batch"].nunique() > 1:
        sc.external.pp.harmony_integrate(adata, key="batch")
        embeddings["Harmony"] = adata.obsm["X_pca_harmony"]

        scvi.settings.seed = 0
        scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch")
        model = scvi.model.SCVI(adata)
        model.train(accelerator="cpu", enable_progress_bar=False)
        embeddings["scVI"] = model.get_latent_representation()
        print(f"scvi-tools {scvi.__version__}: {name}, {model.history['elbo_train'].shape[0]} epochs")
    np.savez(f"{name}_embeddings.npz", obs_names=adata.obs_names.to_numpy(), **embeddings)


if __name__ == "__main__":
    main(sys.argv[1])

Writing integrate.py


In [6]:
for name in ["by_patient", "across_patients", "reference"]:
    !MPLBACKEND=Agg PYTHONWARNINGS=ignore {INTEGRATION_PYTHON} integrate.py {name} 2>> integrate.log

scvi-tools 1.4.2: by_patient, 400 epochs


scvi-tools 1.4.2: across_patients, 400 epochs


## Integration removes the differences between patients when each patient is its own batch

We score each embedding on the 30 nearest neighbors of each cell, through the share of neighbors that carry the same label as the cell.
Batch mixing is read on the microenvironment cells (stromal and normal epithelial cells), whose biology is the same in every patient, so that an integration should mix them. The share of neighbors from their own batch is 1/6 when the six batches are fully mixed and 1 when they are fully separated.
The patient-specific structure is read on the cancer cells, as the share of neighbors from their own patient, searched among the cancer cells, and the share from their own clone, searched among the cancer cells of the same patient.
The reference gives the values the biology sets, which an integration should keep.
We leave out the doublets, whose labels are those of one of their two cells.

In [7]:
def same_label_share(embedding, labels, k=30):
    # mean share of the k nearest neighbors of each cell (itself excluded) that carry its label
    k = min(k, len(labels) - 1)
    neighbors = NearestNeighbors(n_neighbors=k).fit(embedding).kneighbors(return_distance=False)
    labels = np.asarray(labels)
    return (labels[neighbors] == labels[:, None]).mean()


rows = []
for name, adata in data.items():
    obs = adata.obs
    stored = np.load(f"{name.replace(' ', '_')}_embeddings.npz", allow_pickle=True)
    for method in [m for m in ("none", "Harmony", "scVI") if m in stored]:
        emb = pd.DataFrame(stored[method], index=stored["obs_names"]).loc[obs.index].to_numpy()
        single = ~obs["is_doublet"].to_numpy()
        micro = single & obs["cell_type"].isin(["stromal", "epithelial"]).to_numpy()
        cancer = single & (obs["cell_type"] == "cancer").to_numpy()
        clone_shares = [(same_label_share(emb[cancer & (obs["patient"] == p).to_numpy()],
                                          obs.loc[cancer & (obs["patient"] == p).to_numpy(), "lineage"]),
                         (cancer & (obs["patient"] == p).to_numpy()).sum()) for p in obs["patient"].unique()]
        rows.append({"design": name, "integration": method,
                     "own batch (microenvironment)": same_label_share(emb[micro], obs.loc[micro, "batch"])
                     if obs["batch"].nunique() > 1 else np.nan,
                     "own patient (cancer)": same_label_share(emb[cancer], obs.loc[cancer, "patient"]),
                     "own clone (cancer, within patient)": np.average([s for s, _ in clone_shares],
                                                                      weights=[n for _, n in clone_shares])})
scores = pd.DataFrame(rows).set_index(["design", "integration"]).round(2)
scores

own batch (microenvironment)  \
design          integration                                 
by patient      none                                 0.22   
                Harmony                              0.21   
                scVI                                 0.18   
across patients none                                 0.21   
                Harmony                              0.17   
                scVI                                 0.17   
reference       none                                  NaN   

                             own patient (cancer)  \
design          integration                         
by patient      none                         0.92   
                Harmony                      0.39   
                scVI                         0.42   
across patients none                         0.86   
                Harmony                      0.86   
                scVI                         0.86   
reference       none                         0.88   

                             own clone (cancer, within patient)  
design          integration                                      
by patient      none                                       0.55  
                Harmony                                    0.55  
                scVI                                       0.55  
across patients none                                       0.53  
                Harmony                                    0.53  
                scVI                                       0.55  
reference       none                                       0.54

Without integration, the microenvironment cells were already close to mixed in both designs, with 0.22 and 0.21 of their neighbors from their own batch against 1/6 for full mixing, since the batch effect of the 10x preset is mild.
Harmony and scVI brought this share to between 0.17 and 0.21.
In the reference, 88% of the neighbors of each cancer cell came from the same patient, since the tumors of different patients carry different clones.
When each patient was its own batch, Harmony and scVI cut this share to 0.39 and 0.42, removing most of the differences between the tumors together with the batch effect, while the embedding without integration kept them (0.92).
When the patients were spread across batches, both integrations kept the share at 0.86.
Within each patient, the share of neighbors from the same clone stayed at its value in the reference (0.53 to 0.55) in every embedding, so the clonal structure within each tumor was kept in both designs.
Overall, the design decided whether integration kept the differences between patients, which the scores of batch mixing alone do not show.

## Making the benchmark harder or easier

The strength of the batch effect is set by the batch settings of [`scRNA`](../reference/scRNA.md).
A larger per-gene factor (`sigma_batch`) or depth shift (`depth_batch_sigma`) separates the batches more, and per-batch settings (`batch_kwargs` of [`run_scrna_batches`](../reference/scRNA.md)) let each batch take its own depth, dispersion, ambient RNA and doublet rate, as [Shape the data](03_shape_the_data.ipynb) shows.
The design is the main lever.
Confounding batch with patient, as in the first design, asks an integration to tell technical from biological differences between patients, which the data alone can not settle.
More patients, larger tumors (`target_cancer`) and more dissociated cells per patient give each patient more cancer cells and clones, and a more unstable genome (`mutation_rate` and `wgd_rate` in the configuration, see [`GenotypeTumor`](../reference/GenotypeTumor.md)) makes the patients and their clones differ more.